# DFLab on Colab
Select a GPU runtime with Python 3.10–3.12. Run setup once per new runtime. The model and augmentation policy are unchanged. Training runs in an isolated environment, not the notebook kernel.


In [ ]:
import sys, subprocess
from pathlib import Path
assert (3, 10) <= sys.version_info[:2] <= (3, 12), sys.version
if not Path('/content/DFLab/.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/thanhquan123hi1/DFLab.git', '/content/DFLab'], check=True)
%cd /content/DFLab
subprocess.run([sys.executable, 'scripts/setup_colab.py'], check=True)
PYTHON = '/content/DFLab/.venv-colab/bin/python'


## Dataset paths
Mount Drive for checkpoint storage. Edit these three paths to match your uploaded data. Images in JSON must resolve under RGB_DIR (or be valid absolute paths).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RGB_DIR = '/content/DFLab/datasets/rgb'
JSON_DIR = '/content/DFLab/preprocessing/dataset_json'
LOG_DIR = '/content/drive/MyDrive/DFLab/runs'
import json
settings = json.dumps(dict(rgb_dir=RGB_DIR, dataset_json_folder=JSON_DIR, log_dir=LOG_DIR))
script = '''
import sys, json, yaml
from pathlib import Path
settings = json.loads(sys.argv[1])
for name in ('train_config.yaml', 'test_config.yaml'):
    path = Path('training/config') / name
    config = yaml.safe_load(path.read_text())
    config.update({k:v for k,v in settings.items() if k != 'log_dir' or name == 'train_config.yaml'})
    path.write_text(yaml.safe_dump(config, sort_keys=False))
'''
subprocess.run([PYTHON, '-I', '-c', script, settings], check=True)


## Verify and train
The check uses synthetic RGB/JSON. Training uses your real data and downloads CLIP on the first run. Lower batch sizes in the detector YAML if needed for GPU memory.


In [ ]:
subprocess.run([PYTHON, '-I', 'scripts/check_environment.py'], check=True)
subprocess.run([PYTHON, '-I', '-m', 'pytest', 'tests', '-q'], check=True)


In [ ]:
subprocess.run([PYTHON, '-I', 'training/train.py', '--detector_path', 'training/config/detector/biasln.yaml', '--train_dataset', 'FaceForensics++'], check=True)


## Final test
Set WEIGHTS to the selected source-validation checkpoint. This cell is optional until training has completed.


In [ ]:
WEIGHTS = '/content/drive/MyDrive/DFLab/runs/REPLACE_WITH_RUN/validation/FaceForensics++/ckpt_best.pth'
assert Path(WEIGHTS).is_file(), 'Set WEIGHTS to your saved checkpoint'
subprocess.run([PYTHON, '-I', 'training/test.py', '--detector_path', 'training/config/detector/biasln.yaml', '--weights_path', WEIGHTS, '--test_dataset', 'Celeb-DF-v2'], check=True)
